In [5]:
import os
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.mobilenet import preprocess_input
import mediapipe as mp
from collections import deque
import time

try:
    import winsound
except Exception:
    winsound = None



# ======================= Configuration =======================
EYE_MODEL_PATH = '../models/eye_model_finetuned.h5'
VIDEO_PATH = None
INPUT_SOURCE = 0 if VIDEO_PATH is None else VIDEO_PATH
MARGIN_PCT_EYE = 0.35
MARGIN_PX = 10
MIN_SIZE = 16
FREQ = 2500
DUR = 1000

# --- Optimization Settings ---
RESIZE_WIDTH = 640
FRAME_SKIP_RATE = 2
frame_counter = 0

# --- Sigmoid Model Settings ---
EYE_THRESHOLD = 0.5

# ======================= PERCLOS Settings =======================
PERCLOS_WINDOW = 60  # 60 seconds window
PERCLOS_THRESHOLDS = {
    'low': 3.75,           # s <= 3.75%
    'low_high': 8.5,       # 3.75% < s <= 7.5%
    'moderate': 10.0,     # 7.5% < s <= 10.0%
    'moderate_high': 12.0  # 10.0% < s <= 12%
    # severe: s > 12%
}

eye_closure_history = deque()

# ======================= Cooldown Settings =======================
ALERT_COOLDOWN = 6
last_eye_alert = 0

# ======================= Landmarks =======================
LEFT_EYE_IDX = [33, 133, 160, 159, 158, 157, 173, 246]
RIGHT_EYE_IDX = [362, 263, 387, 386, 385, 384, 398, 466]



try:
    eye_model = load_model(EYE_MODEL_PATH, compile=False)
except Exception as e:
    print(f"Error loading models: {e}")
    eye_model = None



# ======================= Helper Functions =======================
def preprocess_robust(img):
    """Robust preprocessing combining reflection removal and CLAHE"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 245, 255, cv2.THRESH_BINARY)
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=1)
    inpainted = cv2.inpaint(img, mask, 5, cv2.INPAINT_TELEA)

    lab = cv2.cvtColor(inpainted, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    l = clahe.apply(l)
    enhanced = cv2.merge([l, a, b])
    enhanced = cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)

    result = cv2.GaussianBlur(enhanced, (3, 3), 0)
    return result

def crop_region(frame, landmarks, idx_list, margin_pct):
    img_h, img_w = frame.shape[:2]
    pts = np.array([(int(landmarks[i].x * img_w), int(landmarks[i].y * img_h)) for i in idx_list])
    x, y, w_box, h_box = cv2.boundingRect(pts)
    margin_x = int(w_box * margin_pct) + MARGIN_PX
    margin_y = int(h_box * margin_pct) + MARGIN_PX
    x1 = max(x - margin_x, 0)
    y1 = max(y - margin_y, 0)
    x2 = min(x + w_box + margin_x, img_w)
    y2 = min(y + h_box + margin_y, img_h)
    crop = frame[y1:y2, x1:x2]
    return crop, (x1, y1, x2, y2)

def process_eye(eye_crop):
    if eye_crop.size == 0:
        return None

    ch, cw = eye_crop.shape[:2]
    if cw < MIN_SIZE or ch < MIN_SIZE:
        return None

    eye_processed = preprocess_robust(eye_crop)
    eye_gray = cv2.cvtColor(eye_processed, cv2.COLOR_BGR2GRAY)
    eye_gray = cv2.cvtColor(eye_gray, cv2.COLOR_GRAY2BGR)

    inp = cv2.resize(eye_gray, (224, 224)).astype(np.float32)
    inp = np.expand_dims(inp, axis=0)
    inp = preprocess_input(inp)

    pred_prob = eye_model.predict(inp, verbose=0)[0][0]

    pred_class = 1 if pred_prob > EYE_THRESHOLD else 0

    return pred_class, pred_prob

def calculate_perclos():
    if len(eye_closure_history) == 0:
        return 0.0

    current_time = time.time()
    cutoff_time = current_time - PERCLOS_WINDOW

    while eye_closure_history and eye_closure_history[0][0] < cutoff_time:
        eye_closure_history.popleft()

    if len(eye_closure_history) == 0:
        return 0.0

    closed_count = sum(1 for _, is_closed in eye_closure_history if is_closed)
    total_count = len(eye_closure_history)

    perclos = (closed_count / total_count) * 100.0 if total_count > 0 else 0.0
    return perclos

def get_drowsiness_level(perclos):
    if perclos <= PERCLOS_THRESHOLDS['low']:
        return "Normal", (0, 255, 0)  # Green
    elif perclos <= PERCLOS_THRESHOLDS['low_high']:
        return "Low Drowsiness", (0, 255, 255)  # Yellow
    elif perclos <= PERCLOS_THRESHOLDS['moderate']:
        return "Moderate Drowsiness", (0, 165, 255)  # Orange
    elif perclos <= PERCLOS_THRESHOLDS['moderate_high']:
        return "Moderate-High Drowsiness", (0, 100, 255)  # Dark Orange
    else:
        return "SEVERE DROWSINESS", (0, 0, 255)  # Red

# ======================= Main =======================
cap = cv2.VideoCapture(INPUT_SOURCE)
if not cap.isOpened():
    raise IOError(f"Cannot open video source: {INPUT_SOURCE}")

fps = cap.get(cv2.CAP_PROP_FPS)
if fps is None or fps == 0 or np.isnan(fps):
    delay = 1
else:
    delay = int(max(1, round(1000.0 / fps)))

font = cv2.FONT_HERSHEY_SIMPLEX

mp_face_mesh = mp.solutions.face_mesh

with mp_face_mesh.FaceMesh(static_image_mode=False,
                           max_num_faces=1,
                           refine_landmarks=True,
                           min_detection_confidence=0.5,
                           min_tracking_confidence=0.5) as face_mesh:

    lm = None
    pred_left_eye = None
    pred_right_eye = None
    left_prob = 0.0
    right_prob = 0.0
    left_eye_bbox = (0, 0, 0, 0)
    right_eye_bbox = (0, 0, 0, 0)
    # mouth_bbox = (0, 0, 0, 0)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if RESIZE_WIDTH is not None and frame.shape[1] > RESIZE_WIDTH:
            scale_factor = RESIZE_WIDTH / frame.shape[1]
            frame = cv2.resize(frame, (RESIZE_WIDTH, int(frame.shape[0] * scale_factor)))

        img_h, img_w = frame.shape[:2]
        frame_counter += 1

        if frame_counter % FRAME_SKIP_RATE == 0:
            frame_counter = 0

            img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(img_rgb)

            pred_left_eye = None
            pred_right_eye = None
            left_prob = 0.0
            right_prob = 0.0

            if results.multi_face_landmarks:
                lm = results.multi_face_landmarks[0].landmark

                if eye_model:
                    # --- Left Eye ---
                    left_eye_crop, left_eye_bbox = crop_region(frame, lm, LEFT_EYE_IDX, MARGIN_PCT_EYE)
                    result = process_eye(left_eye_crop)
                    if result is not None:
                        pred_left_eye, left_prob = result

                    # --- Right Eye ---
                    right_eye_crop, right_eye_bbox = crop_region(frame, lm, RIGHT_EYE_IDX, MARGIN_PCT_EYE)
                    result = process_eye(right_eye_crop)
                    if result is not None:
                        pred_right_eye, right_prob = result



        # Draw bounding boxes
        lex1, ley1, lex2, ley2 = left_eye_bbox
        rex1, rey1, rex2, rey2 = right_eye_bbox
        # mx1, my1, mx2, my2 = mouth_bbox

        if lm:
            if pred_left_eye is not None:
                cv2.rectangle(frame, (lex1, ley1), (lex2, ley2), (255, 0, 0), 2)
            if pred_right_eye is not None:
                cv2.rectangle(frame, (rex1, rey1), (rex2, rey2), (0, 255, 0), 2)

        detection_mode = ""
        is_eyes_closed = False

        left_detected = pred_left_eye is not None
        right_detected = pred_right_eye is not None

        if not left_detected and not right_detected:
            eye_label = "No Face"
            eye_color = (0, 255, 255)
            is_eyes_closed = False
            detection_mode = "No Detection"

        elif left_detected and right_detected:
            left_open = (pred_left_eye == 1)
            right_open = (pred_right_eye == 1)
            both_closed = not left_open and not right_open

            detection_mode = "Both Eyes"
            is_eyes_closed = both_closed

            if both_closed:
                eye_label = "Closed Eyes"
                eye_color = (0, 0, 255)
            else:
                eye_label = "Open Eyes"
                eye_color = (0, 255, 0)

            left_status = f"L:Open({left_prob:.2f})" if left_open else f"L:Closed({left_prob:.2f})"
            right_status = f"R:Open({right_prob:.2f})" if right_open else f"R:Closed({right_prob:.2f})"
            cv2.putText(frame, f"{left_status} | {right_status}", (30, 270),
                       font, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

        elif left_detected and not right_detected:
            left_open = (pred_left_eye == 1)
            detection_mode = "Left Eye Only"
            is_eyes_closed = not left_open

            if not left_open:
                eye_label = f"Closed Eye (Left)"
                eye_color = (0, 0, 255)
            else:
                eye_label = f"Open Eye (Left)"
                eye_color = (0, 255, 0)

            cv2.putText(frame, f"L:Detected({left_prob:.2f}) | R:Not Detected", (30, 270),
                       font, 0.5, (255, 165, 0), 1, cv2.LINE_AA)

        else:
            right_open = (pred_right_eye == 1)
            detection_mode = "Right Eye Only"
            is_eyes_closed = not right_open

            if not right_open:
                eye_label = f"Closed Eye (Right)"
                eye_color = (0, 0, 255)
            else:
                eye_label = f"Open Eye (Right)"
                eye_color = (0, 255, 0)

            cv2.putText(frame, f"L:Not Detected | R:Detected({right_prob:.2f})", (30, 270),
                       font, 0.5, (255, 165, 0), 1, cv2.LINE_AA)

        # Record eye closure state with timestamp
        current_time = time.time()
        eye_closure_history.append((current_time, is_eyes_closed))

        # Calculate PERCLOS
        perclos = calculate_perclos()
        drowsiness_level, level_color = get_drowsiness_level(perclos)

        # Display detection mode
        cv2.putText(frame, f"Mode: {detection_mode}", (30, 200),
                   font, 0.7, (255, 255, 0), 1, cv2.LINE_AA)

        # Display current eye status
        cv2.putText(frame, eye_label, (30, 40), font, 1.0, eye_color, 2, cv2.LINE_AA)

        # Display PERCLOS value
        cv2.putText(frame, f"PERCLOS: {perclos:.2f}%", (30, 80),
                   font, 0.9, (255, 255, 255), 2, cv2.LINE_AA)

        # Display drowsiness level
        cv2.putText(frame, f"Level: {drowsiness_level}", (30, 120),
                   font, 0.9, level_color, 2, cv2.LINE_AA)

        # Alert for severe drowsiness
        if perclos > PERCLOS_THRESHOLDS['low_high']:
            cv2.putText(frame, "!!! DROWSINESS ALERT !!!", (30, 160),
                       font, 1.0, (0, 0, 255), 2, cv2.LINE_AA)
            if winsound and (time.time() - last_eye_alert > ALERT_COOLDOWN):
                try:
                    winsound.Beep(FREQ, DUR)
                except:
                    pass
                last_eye_alert = time.time()

        # Display history window size
        cv2.putText(frame, f"Window: {len(eye_closure_history)} samples", (30, 230),
                   font, 0.6, (200, 200, 200), 1, cv2.LINE_AA)

        cv2.imshow("Driver Monitoring - PERCLOS", frame)
        if cv2.waitKey(delay) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()